<a href="https://colab.research.google.com/github/singh-himanshu3/CV_Project/blob/main/CV_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [ ]:
# Cell 1: Environment Setup and Downloading Pre-trained Weights
import os


!git clone https://github.com/cszn/SCUNet.git
%cd SCUNet


!pip install timm


os.makedirs('model_zoo', exist_ok=True)


!wget https://github.com/cszn/SCUNet/releases/download/v1.0/scunet_color_real_psnr.pth -P model_zoo/


import torch
if torch.cuda.is_available():
    print(f" Success! GPU active: {torch.cuda.get_device_name(0)}")
    print("SCUNet Repo cloned and weights downloaded.")
else:
    print(" ERROR: No GPU found. Go to Runtime > Change runtime type and select T4 GPU.")

In [ ]:
!pip install thop einops

In [ ]:
# Cell 2: High-Confidence Forensic Engine
import torch
import numpy as np
import cv2
from models.network_scunet import SCUNet
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SCUNet(in_nc=3, config=[4,4,4,4,4,4,4], dim=64)
model.load_state_dict(torch.load('/content/drive/MyDrive/scunet_cctv_weather_finetuned.pth'), strict=True)
model.eval().to(device)

def simple_tiled_inference(img_tensor, model, window_size=256, overlap=64):
    b, c, h, w = img_tensor.shape
    stride = window_size - overlap


    pad_h = (stride - (h - window_size) % stride) % stride if h > window_size else window_size - h
    pad_w = (stride - (w - window_size) % stride) % stride if w > window_size else window_size - w
    img_p = torch.nn.functional.pad(img_tensor, (0, pad_w, 0, pad_h), mode='reflect')

    _, _, ph, pw = img_p.shape
    out = torch.zeros_like(img_p)
    weights = torch.zeros_like(img_p)


    w1d = np.hanning(window_size)
    window = torch.from_numpy(np.outer(w1d, w1d)).float().to(device).unsqueeze(0).unsqueeze(0)

    for y in range(0, ph - window_size + 1, stride):
        for x in range(0, pw - window_size + 1, stride):
            patch = img_p[:, :, y:y+window_size, x:x+window_size]
            with torch.no_grad():
                res = model(patch)
            out[:, :, y:y+window_size, x:x+window_size] += res * window
            weights[:, :, y:y+window_size, x:x+window_size] += window

    return (out / (weights + 1e-8))[:, :, :h, :w].clamp(0, 1)

print(" High-Confidence Engine Online.")

In [ ]:
# Cell 3: Hybrid CV Post-Processing Head (Interactive)
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

cv_img = (denoised_img * 255).astype(np.uint8)

def apply_hybrid_head(d=5, sigma_color=50, sigma_space=50, sharpen_amount=1.5):

    filtered = cv2.bilateralFilter(cv_img, d, sigma_color, sigma_space)

    blurred = cv2.GaussianBlur(filtered, (0, 0), 3)
    sharpened = cv2.addWeighted(filtered, 1.0 + sharpen_amount, blurred, -sharpen_amount, 0)

    sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)

    fig, ax = plt.subplots(1, 3, figsize=(18, 6))

    ax[0].imshow(cv_img)
    ax[0].set_title("1. Base AI Output (Soft/Smooth)")
    ax[0].axis('off')

    ax[1].imshow(filtered)
    ax[1].set_title("2. Bilateral Filter")
    ax[1].axis('off')

    ax[2].imshow(sharpened)
    ax[2].set_title(f"3. Final Forensic Output (Sharpened)")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()

print(" Adjust the sliders below to tune the forensic output!")

interact(apply_hybrid_head,
         d=IntSlider(min=1, max=15, step=2, value=5, description='Bilateral d:'),
         sigma_color=IntSlider(min=10, max=150, step=5, value=50, description='Sigma Color:'),
         sigma_space=IntSlider(min=10, max=150, step=5, value=50, description='Sigma Space:'),
         sharpen_amount=FloatSlider(min=0.0, max=5.0, step=0.1, value=1.5, description='Sharpening:'))

In [ ]:
# Cell 4: Temporal Video Denoising Pipeline (Corrected Lighting)
import cv2
import numpy as np
import torch
from google.colab import files
from tqdm.notebook import tqdm

print("PLEASE UPLOAD A SHORT TEST VIDEO (.mp4, .avi) ")
print("(Pro-tip: Keep it under 5 seconds for this initial test to save time!)")
uploaded_video = files.upload()

if uploaded_video:
    video_filename = list(uploaded_video.keys())[0]
    output_filename = "forensic_enhanced_output.mp4"


    cap = cv2.VideoCapture(video_filename)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_filename, fourcc, fps, (width, height))

    print(f"\nProcessing Video: {video_filename}")
    print(f"Resolution: {width}x{height} | Total Frames: {total_frames} | FPS: {fps}")

    for i in tqdm(range(total_frames), desc="Denoising & Enhancing Frames"):
        ret, frame = cap.read()
        if not ret:
            break

        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img_tensor = torch.from_numpy(np.transpose(rgb_frame, (2, 0, 1))).float().unsqueeze(0) / 255.0
        img_tensor = img_tensor.to(device)

        denoised_tensor = simple_tiled_inference(img_tensor, model, window_size=256, overlap=32)

        denoised_img = denoised_tensor.squeeze().cpu().numpy()
        denoised_img = np.transpose(denoised_img, (1, 2, 0))
        cv_img = (denoised_img * 255).astype(np.uint8)
        cv_img = cv2.cvtColor(cv_img, cv2.COLOR_RGB2BGR)

        filtered = cv2.bilateralFilter(cv_img, d=5, sigmaColor=50, sigmaSpace=50)

        lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)

        l_blur = cv2.GaussianBlur(l, (0, 0), 3)
        l_sharp = cv2.addWeighted(l, 1.5, l_blur, -0.5, 0)


        lab_merged = cv2.merge((l_sharp, a, b))
        final_frame = cv2.cvtColor(lab_merged, cv2.COLOR_LAB2BGR)

        out.write(final_frame)

    cap.release()
    out.release()

    print(f"\n Video processing complete! Saved as '{output_filename}'")
    print(" Triggering automatic download to your computer...")
    files.download(output_filename)
else:
    print("No video uploaded. Please try again.")

In [ ]:
# Cell 5: Live Webcam Forensic Feed (Near-Real-Time Snapshot)
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

def take_photo(filename='webcam_capture.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      const btn = document.createElement('button');
      btn.textContent = ' Capture & Enhance';
      div.appendChild(btn);

      await new Promise((resolve) => btn.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

print(" Starting Webcam... Click 'Capture & Enhance' when ready! ")
try:
  capture_path = take_photo()

  print("Processing live frame through the SCUNet Tiled Engine...")

  live_img = cv2.imread(capture_path)
  live_img_rgb = cv2.cvtColor(live_img, cv2.COLOR_BGR2RGB)


  np.random.seed(42)
  noise = np.random.normal(0, 40, live_img_rgb.shape)
  noisy_live_img = np.clip(live_img_rgb + noise, 0, 255).astype(np.uint8)


  img_tensor = torch.from_numpy(np.transpose(noisy_live_img, (2, 0, 1))).float().unsqueeze(0) / 255.0
  img_tensor = img_tensor.to(device)


  denoised_tensor = simple_tiled_inference(img_tensor, model, window_size=256, overlap=64)
  denoised_img = denoised_tensor.squeeze().cpu().numpy()
  denoised_img = np.transpose(denoised_img, (1, 2, 0))


  cv_out_8 = (denoised_img * 255).astype(np.uint8)


  cleaned = cv2.fastNlMeansDenoisingColored(cv_out_8, None, h=6, hColor=6)


  lab = cv2.cvtColor(cleaned, cv2.COLOR_RGB2LAB)
  l, a, b = cv2.split(lab)
  l_blur = cv2.GaussianBlur(l, (0, 0), 2)
  l = cv2.addWeighted(l, 1.6, l_blur, -0.6, 0)
  final_forensic_rgb = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


  fig, ax = plt.subplots(1, 3, figsize=(18, 6))
  ax[0].imshow(noisy_live_img)
  ax[0].set_title("1. Live Webcam (With Simulated Noise)")
  ax[0].axis('off')

  ax[1].imshow(denoised_img)
  ax[1].set_title("2. SCUNet Base AI Output")
  ax[1].axis('off')

  ax[2].imshow(final_forensic_rgb)
  ax[2].set_title("3. Final Forensic Output")
  ax[2].axis('off')

  plt.tight_layout()
  plt.show()

except Exception as err:
  print(f"Webcam capture failed: {err}")

In [ ]:
# Cell: All-Weather Forensic Analysis (AI + Median + NLM)
from google.colab import files
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    img = cv2.imread(filename)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    print(" Plucking out Salt & Pepper noise spikes...")
    img_pre = cv2.medianBlur(img_rgb, 3)


    print("Running AI structural reconstruction...")
    itensor = torch.from_numpy(np.transpose(img_pre, (2,0,1))).float().unsqueeze(0).to(device) / 255.0
    denoised_tensor = simple_tiled_inference(itensor, model)
    ai_out = np.transpose(denoised_tensor.squeeze().cpu().numpy(), (1,2,0))
    ai_out_8 = (ai_out * 255).astype(np.uint8)


    print("Polishing AI artifacts with Non-Local Means...")
    cleaned = cv2.fastNlMeansDenoisingColored(ai_out_8, None, h=8, hColor=8, templateWindowSize=7, searchWindowSize=21)


    lab = cv2.cvtColor(cleaned, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)


    l_blur = cv2.GaussianBlur(l, (0, 0), 2)
    l = cv2.addWeighted(l, 1.7, l_blur, -0.7, 0)

    final_rgb = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)


    fig, ax = plt.subplots(1, 2, figsize=(20, 10))
    ax[0].imshow(img_rgb); ax[0].set_title("1. Raw Input (Noisy)"); ax[0].axis('off')
    ax[1].imshow(final_rgb); ax[1].set_title("2. Final Forensic Reconstruction"); ax[1].axis('off')
    plt.show()


    cv2.imwrite("Forensic_Final_Cleaned.png", cv2.cvtColor(final_rgb, cv2.COLOR_RGB2BGR))
    files.download("Forensic_Final_Cleaned.png")